In [ ]:
import pandas as pd

train = pd.read_csv("data/train_test.csv")
final = pd.read_csv("data/validation.csv")
template = pd.read_csv("data/validation_predictions_template.csv")
december = pd.read_csv("data/december_chart_inputs.csv")

for name, df in [("Training data", train), ("Final prediction data", final)]:
    print(f"\n--- {name} ---")
    print("Rows and columns:", df.shape)
    print("Date range:", df["date"].min(), "to", df["date"].max())
    print("Duplicate IDs:", df["load_id"].duplicated().sum())
    print("Invalid weights:", (df["weight"] <= 0).sum())
    print("Missing values:")
    print(df.isna().sum())

print(
    "\nPrediction template IDs match:",
    set(template["load_id"]) == set(final["load_id"])
)

print("December chart rows:", len(december))
print("December columns:", december.columns.tolist())

In [ ]:
development = train.copy()
development["date"] = pd.to_datetime(development["date"])

july_start = pd.Timestamp("2025-07-01")
september_start = pd.Timestamp("2025-09-01")

train_split = development[
    development["date"] < july_start
].copy()

validation_split = development[
    (development["date"] >= july_start)
    & (development["date"] < september_start)
].copy()

test_split = development[
    development["date"] >= september_start
].copy()

assert (
    len(train_split) + len(validation_split) + len(test_split)
    == len(development)
)

assert train_split["date"].max() < validation_split["date"].min()
assert validation_split["date"].max() < test_split["date"].min()

for name, split in [
    ("Training", train_split),
    ("Validation", validation_split),
    ("Local test", test_split),
]:
    print(
        f"{name}: {len(split):,} rows | "
        f"{split['date'].min().date()} to "
        f"{split['date'].max().date()}"
    )

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def prepare_features(data):
    features = data[
        ["pickup", "delivery", "distance", "equipment", "weight", "date"]
    ].copy()

    dates = pd.to_datetime(features.pop("date"))

    features["weight"] = features["weight"].mask(
        features["weight"] <= 0
    )

    features["month"] = dates.dt.month
    features["day_of_week"] = dates.dt.dayofweek

    return features

X_train = prepare_features(train_split)
X_validation = prepare_features(validation_split)

y_train = train_split["posted_rate"].copy()
y_validation = validation_split["posted_rate"].copy()
y_test = test_split["posted_rate"].copy()

In [ ]:
numeric_columns = [
    "distance",
    "weight",
    "month",
    "day_of_week"
]

categorical_columns = [
    "pickup",
    "delivery",
    "equipment"
]

preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median", add_indicator=True),
        numeric_columns
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        categorical_columns
    )
])

X_train_ready = preprocessor.fit_transform(X_train)
X_validation_ready = preprocessor.transform(X_validation)


In [ ]:
for name, values in [
    ("Training", X_train_ready),
    ("Validation", X_validation_ready)
]:
    assert np.isfinite(values).all()
    print(name, values.shape)

print("Training weight median:", X_train["weight"].median())


In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

baseline_rate_per_mile = (
    train_split["posted_rate"] / train_split["distance"]
).median()

baseline_validation_predictions = (
    validation_split["distance"] * baseline_rate_per_mile
)

baseline_mae = mean_absolute_error(
    y_validation,
    baseline_validation_predictions
)

baseline_rmse = root_mean_squared_error(
    y_validation,
    baseline_validation_predictions
)

print(f"Typical price per mile: ${baseline_rate_per_mile:.4f}")
print(f"Validation MAE: ${baseline_mae:.2f}")
print(f"Validation RMSE: ${baseline_rmse:.2f}")

In [ ]:
from pathlib import Path

validation_results = pd.DataFrame([
    {
        "model": "Price-per-mile baseline",
        "mae": baseline_mae,
        "rmse": baseline_rmse
    }
])

Path("reports").mkdir(exist_ok=True)




In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

model.fit(X_train_ready, y_train)

print("Training complete")

In [ ]:
model_validation_predictions = model.predict(X_validation_ready)

model_mae = mean_absolute_error(
    y_validation,
    model_validation_predictions
)

model_rmse = root_mean_squared_error(
    y_validation,
    model_validation_predictions
)

mae_improvement = (
    (baseline_mae - model_mae) / baseline_mae
) * 100

print(f"Baseline MAE: ${baseline_mae:.2f}")
print(f"Model MAE: ${model_mae:.2f}")
print(f"Baseline RMSE: ${baseline_rmse:.2f}")
print(f"Model RMSE: ${model_rmse:.2f}")
print(f"MAE improvement: {mae_improvement:.2f}%")

In [ ]:
model_name = "Histogram gradient boosting"

model_result = pd.DataFrame([
    {
        "model": model_name,
        "mae": model_mae,
        "rmse": model_rmse
    }
])

validation_results = pd.concat(
    [
        validation_results[
            validation_results["model"] != model_name
        ],
        model_result
    ],
    ignore_index=True
)

validation_results.to_csv(
    "reports/validation_results.csv",
    index=False
)

validation_results.sort_values("mae")

In [ ]:
validation_errors = validation_split[
    [
        "load_id",
        "date",
        "pickup",
        "delivery",
        "equipment",
        "distance",
        "weight",
        "posted_rate"
    ]
].copy()

validation_errors["predicted_rate"] = model_validation_predictions

validation_errors["absolute_error"] = (
    validation_errors["posted_rate"]
    - validation_errors["predicted_rate"]
).abs()

validation_errors["baseline_absolute_error"] = (
    validation_errors["posted_rate"]
    - baseline_validation_predictions
).abs()

validation_errors["month"] = (
    validation_errors["date"].dt.strftime("%Y-%m")
)

In [ ]:
monthly_errors = validation_errors.groupby("month").agg(
    shipments=("load_id", "size"),
    model_mae=("absolute_error", "mean"),
    baseline_mae=("baseline_absolute_error", "mean")
)

monthly_errors.round(2)

In [ ]:
equipment_errors = validation_errors.groupby("equipment").agg(
    shipments=("load_id", "size"),
    model_mae=("absolute_error", "mean"),
    baseline_mae=("baseline_absolute_error", "mean")
)

equipment_errors.round(2)

In [ ]:
validation_errors.to_csv(
    "reports/validation_errors.csv",
    index=False
)

validation_errors.nlargest(10, "absolute_error")[
    [
        "date",
        "pickup",
        "delivery",
        "distance",
        "posted_rate",
        "predicted_rate",
        "absolute_error"
    ]
].round(2)

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline

evaluation_training = pd.concat(
    [train_split, validation_split],
    ignore_index=True
)

evaluation_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", clone(model))
])

evaluation_pipeline.fit(
    prepare_features(evaluation_training),
    evaluation_training["posted_rate"]
)

print("Refitting complete")


In [ ]:
test_predictions = evaluation_pipeline.predict(
    prepare_features(test_split)
)

evaluation_baseline_rate = (
    evaluation_training["posted_rate"]
    / evaluation_training["distance"]
).median()

test_baseline_predictions = (
    test_split["distance"] * evaluation_baseline_rate
)

test_results = pd.DataFrame([
    {
        "model": "Price-per-mile baseline",
        "mae": mean_absolute_error(
            y_test,
            test_baseline_predictions
        ),
        "rmse": root_mean_squared_error(
            y_test,
            test_baseline_predictions
        )
    },
    {
        "model": "Histogram gradient boosting",
        "mae": mean_absolute_error(
            y_test,
            test_predictions
        ),
        "rmse": root_mean_squared_error(
            y_test,
            test_predictions
        )
    }
])

test_results.to_csv(
    "reports/local_test_results.csv",
    index=False
)

test_results.round(2)

In [ ]:
test_errors = test_split[
    ["load_id", "date", "equipment", "posted_rate"]
].copy()

test_errors["predicted_rate"] = test_predictions

test_errors["absolute_error"] = (
    test_errors["posted_rate"]
    - test_errors["predicted_rate"]
).abs()

test_errors["month"] = test_errors["date"].dt.strftime("%Y-%m")

test_errors.to_csv(
    "reports/local_test_errors.csv",
    index=False
)

test_errors.groupby("month").agg(
    shipments=("load_id", "size"),
    mae=("absolute_error", "mean")
).round(2)

In [ ]:
final_pipeline = clone(evaluation_pipeline)

final_pipeline.fit(
    prepare_features(development),
    development["posted_rate"]
)

print("Final training complete")

In [ ]:
assert final["load_id"].is_unique
assert template["load_id"].is_unique

prediction_lookup = pd.Series(
    final_pipeline.predict(prepare_features(final)),
    index=final["load_id"]
)

submission = template[["load_id"]].copy()

submission["predicted_rate"] = submission["load_id"].map(
    prediction_lookup
)

assert len(submission) == 12000
assert set(submission["load_id"]) == set(final["load_id"])
assert np.isfinite(submission["predicted_rate"]).all()
assert (submission["predicted_rate"] > 0).all()

submission.to_csv(
    "validation_predictions.csv",
    index=False
)

december_output = december.copy()

december_output["predicted_rate"] = final_pipeline.predict(
    prepare_features(december_output)
)

assert len(december_output) == 31
assert np.isfinite(december_output["predicted_rate"]).all()
assert (december_output["predicted_rate"] > 0).all()

december_output.to_csv(
    "december_predictions.csv",
    index=False
)

print("Saved validation_predictions.csv")
print("Saved december_predictions.csv")

submission.head().round(2)

In [ ]:
import subprocess
import sys
from IPython.display import Image, display

checker_result = subprocess.run(
    [
        sys.executable,
        "score.py",
        "--predictions",
        "validation_predictions.csv",
        "--december-predictions",
        "december_predictions.csv"
    ],
    capture_output=True,
    text=True
)

print(checker_result.stdout)

if checker_result.returncode != 0:
    print(checker_result.stderr)
    raise RuntimeError("Submission checker failed")

display(
    Image(filename="scorer_results/candidate_december.png")
)

In [ ]:
import joblib

Path("models").mkdir(exist_ok=True)

joblib.dump(
    final_pipeline,
    "models/freight_model.joblib"
)

print("Model saved")